# Detection experiments — D01 to D08

This notebook compares the saved detection studies: statistical and unsupervised baselines, supervised learning, feature sensitivity, and season generalisation. It does not retrain models when run.

## Study map

- **D01–D05:** detection comparisons and feature-sensitivity studies.
- **D06:** classical seasonal generalisation.
- **D07:** selected XGBoost primary detector.
- **D08:** XGBoost seasonal generalisation.

To reproduce any study, run `sfsa train` with the matching configuration in `configs/experiments/`; for example D07 uses `D07_xgboost_detection.yaml`.

In [4]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results' / 'final_runs'
STUDIES = [
    'D01_baseline', 'D02_classical', 'D03_supervised',
    'D04a_control_inclusive', 'D04b_control_weather', 'D05_temporal',
    'D06_classical_season_generalisation', 'D07_xgboost_detection',
    'D08_xgboost_season_generalisation',
]

def latest_run(experiment):
    matches = sorted(RESULTS.glob(f'*_{experiment}'))
    if not matches:
        raise FileNotFoundError(f'No completed run found for {experiment}.')
    return matches[-1]

In [5]:
rows = []
for study in STUDIES:
    run_dir = latest_run(study)
    metrics = pd.read_csv(run_dir / 'metrics_summary.csv')
    metrics = metrics[metrics['metric'].isin(['f1', 'precision', 'recall', 'fpr', 'false_alarms_per_day'])]
    metrics['study'] = study.split('_', 1)[0]
    metrics['model'] = metrics['model'].replace({'supervised': 'Histogram Gradient Boosting'})
    rows.append(metrics)

comparison = pd.concat(rows, ignore_index=True)
table = comparison.pivot_table(
    index=['study', 'model'], columns='metric', values='mean', aggfunc='first'
).reset_index()
display(table.sort_values(['study', 'model']).round(4))

metric,study,model,f1,false_alarms_per_day,fpr,precision,recall
0,D01,isolation_forest,0.3832,126.7805,0.0880,0.7302,0.2877
1,D01,statistical,0.9765,72.7114,0.0505,0.9558,1.0000
2,D02,isolation_forest,0.3434,144.1169,0.1001,0.7053,0.2427
3,D02,one_class_svm,0.9567,132.2315,0.0918,0.9181,1.0000
4,D03,Histogram Gradient Boosting,0.9940,17.8171,0.0124,0.9885,1.0000
5,D04a,isolation_forest,0.5040,119.4472,0.0829,0.8006,0.4014
6,D04a,statistical,0.9735,81.6911,0.0567,0.9500,1.0000
7,D04b,isolation_forest,0.4024,108.9024,0.0756,0.7640,0.3135
8,D04b,statistical,0.9734,81.8252,0.0568,0.9499,1.0000
9,D05,isolation_forest,0.4455,167.4756,0.1163,0.7185,0.3439


## Interpretation guardrails

D07 is selected as the primary binary detector. Its strong same-dataset result must be read alongside D08: controlled fault sessions from one facility do not establish deployment performance in a new building. The full-day source labels also mean a reported zero detection delay is not verified physical fault-onset time.

In [6]:
for study in ['D06_classical_season_generalisation', 'D08_xgboost_season_generalisation']:
    run_dir = latest_run(study)
    print(f'\n{study}: {run_dir.name}')
    by_fold = pd.read_csv(run_dir / 'metrics_by_fold.csv')
    display(by_fold[by_fold['metric'].isin(['f1', 'precision', 'recall', 'fpr'])].round(4))


D06_classical_season_generalisation: 20261026T141718Z_rbc_ashrae1312_real_D06_classical_season_generalisation


KeyError: 'metric'